# Submissao 2 - DistilBERT

Fine-tuning de DistilBERT com os melhores hiperparâmetros e predição sobre `subm2.csv`.

**Modelo:** DistilBERT (distilbert-base-uncased) fine-tuned

**Melhores params:** lr=3e-5, batch_size=16, epochs=3, max_len=128

In [ ]:
import sys
import os
from pathlib import Path

sys.path.append(os.path.abspath('..'))
os.environ['HSA_OVERRIDE_GFX_VERSION'] = '12.0.0'

import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from src.models_pytorch.distilbert import DistilBERTClassifier, DistilBERTDataset, get_tokenizer

train_csv = Path('../data/processed/dataset_combined.csv')
input_csv = Path('validation_data/subm2.csv')
output_csv = Path('subm2-g8-MEI-B.csv')

CLASSES = ["Anthropic", "Google", "Human", "Meta", "OpenAI"]
LABEL_MAP = {label: i for i, label in enumerate(CLASSES)}
IDX_TO_LABEL = {i: label for label, i in LABEL_MAP.items()}
NUM_CLASSES = len(CLASSES)
SEED = 42

LR = 3e-5
BATCH_SIZE = 16
MAX_LEN = 128
EPOCHS = 3

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
df_train = pd.read_csv(train_csv, sep=';')
df_train = df_train[df_train['Label'].isin(CLASSES)].copy()
df_train['label_id'] = df_train['Label'].map(LABEL_MAP)

print(f"Train samples: {len(df_train)}")
print(f"Label distribution:\n{df_train['Label'].value_counts()}")

texts_all = list(df_train['Text'])
y_all = df_train['label_id'].values.astype(np.int64)

counts = np.bincount(y_all, minlength=NUM_CLASSES)
weights = 1.0 / np.maximum(counts, 1)
weights = weights / weights.sum() * NUM_CLASSES
class_weights = torch.tensor(weights, dtype=torch.float32)
print(f"Class weights: {class_weights.numpy()}")

texts_tr, texts_val, y_tr, y_val = train_test_split(
    texts_all, y_all, test_size=0.2, random_state=SEED, stratify=y_all
)
print(f"Train: {len(texts_tr)}, Val: {len(texts_val)}")

tokenizer = get_tokenizer()
print("Tokenizer loaded.")

In [ ]:
model = DistilBERTClassifier(output_dim=NUM_CLASSES, dropout=0.3, freeze_bert=False).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

train_ds = DistilBERTDataset(texts_tr, y_tr, tokenizer, max_len=MAX_LEN)
val_ds = DistilBERTDataset(texts_val, y_val, tokenizer, max_len=MAX_LEN)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE)

best_val_acc = 0
best_state = None

t0 = time.time()
for epoch in range(EPOCHS):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch_idx, batch in enumerate(train_dl):
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['label'].to(DEVICE)

        optimizer.zero_grad()
        out = model(input_ids, attention_mask)
        loss = criterion(out, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item() * len(labels)
        correct += (out.argmax(dim=1) == labels).sum().item()
        total += len(labels)

        if (batch_idx + 1) % 50 == 0:
            print(f"    Epoch {epoch+1} batch {batch_idx+1}/{len(train_dl)} "
                  f"loss={total_loss/total:.4f} acc={correct/total:.4f}")

    train_acc = correct / total

    # Validate
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in val_dl:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['label'].to(DEVICE)
            out = model(input_ids, attention_mask)
            correct += (out.argmax(dim=1) == labels).sum().item()
            total += len(labels)
    val_acc = correct / total

    elapsed = time.time() - t0
    print(f"  Epoch {epoch+1}/{EPOCHS} — "
          f"loss={total_loss/total:.4f} train_acc={train_acc:.4f} val_acc={val_acc:.4f} "
          f"({elapsed:.0f}s)")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

print(f"\nBest internal val accuracy: {best_val_acc:.4f}")
model.load_state_dict(best_state)
model.to(DEVICE)

In [ ]:
# --- Predict on submission data ---
df_sub = pd.read_csv(input_csv, sep=';')
print(f"Submission samples: {len(df_sub)}")

texts_sub = list(df_sub['Text'].fillna(''))
y_dummy = np.zeros(len(texts_sub), dtype=np.int64)

sub_ds = DistilBERTDataset(texts_sub, y_dummy, tokenizer, max_len=MAX_LEN)
sub_dl = DataLoader(sub_ds, batch_size=BATCH_SIZE)

model.eval()
all_preds = []
with torch.no_grad():
    for batch in sub_dl:
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        out = model(input_ids, attention_mask)
        all_preds.extend(out.argmax(dim=1).cpu().numpy())

df_sub['Label'] = [IDX_TO_LABEL[int(i)] for i in all_preds]
df_out = df_sub[['ID', 'Label']]
df_out.to_csv(output_csv, sep=';', index=False)

print(f"\nSubmissao guardada em: {output_csv}")
print(f"Distribuicao:\n{df_out['Label'].value_counts()}")
print(df_out.head(10))